# Facial AU Extraction (Colab GPU)

Parallel run to Mac (CPU) and PC (GTX 1660 SUPER) extractions for cross-platform validation.

**Session being processed**: `mouse_longwork_20260513T181315Z`  
**Configuration matches local pipeline**:
- Face model: RetinaFace
- Landmark: MobileFaceNet
- AU: XGBoost
- Emotion: ResMaskNet
- Facepose: img2pose
- Identity: None (disabled to avoid memory issues)
- Subsample: 4 fps

**Before running this notebook**:
1. Runtime → Change runtime type → Hardware accelerator: **GPU (T4)**
2. Upload `recording5m.mp4` to your Google Drive in `/My Drive/AUbatch/incoming/mouse_longwork_20260513T181315Z/`
3. Run cells sequentially

## Cell 1: Verify GPU and check environment

In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Cell 2: Mount Google Drive

Authorize when prompted. Drive will appear at `/content/drive/MyDrive/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify the recording exists
import os
SESSION_DIR = '/content/drive/MyDrive/AUbatch/incoming/mouse_longwork_20260513T181315Z'
VIDEO_PATH = f'{SESSION_DIR}/recording5m.mp4'

if os.path.exists(VIDEO_PATH):
    size_mb = os.path.getsize(VIDEO_PATH) / (1024*1024)
    print(f'Found: {VIDEO_PATH}')
    print(f'Size: {size_mb:.1f} MB')
else:
    print(f'NOT FOUND: {VIDEO_PATH}')
    print('Please upload recording5m.mp4 to your Drive at:')
    print(f'  {SESSION_DIR}/')
    print('Then re-run this cell.')

## Cell 3: Install py-feat with the same pinned versions

This takes ~3-5 minutes on first run. We use the same dependency pins discovered during local setup.

In [ ]:
!pip install -q py-feat
!pip install -q 'scipy<1.14'
!pip install -q 'opencv-python==4.8.1.78'
!pip install -q 'numpy<1.24' --force-reinstall

# Verify
import numpy, cv2, scipy, torch
print(f'numpy {numpy.__version__}')
print(f'cv2 {cv2.__version__}')
print(f'scipy {scipy.__version__}')
print(f'torch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')

## Cell 4: Initialize detector and download model weights

First run downloads ~700MB of model weights. Cached for subsequent runs in this session.

In [ ]:
import torch
torch.set_grad_enabled(False)

from feat import Detector

print('Initializing detector with RetinaFace + XGBoost...')
detector = Detector(
    face_model='retinaface',
    landmark_model='mobilefacenet',
    au_model='xgb',
    emotion_model='resmasknet',
    facepose_model='img2pose',
    identity_model=None,  # Skip identity to avoid memory issues
    device='cuda',
)
print('Detector ready.')

## Cell 5: Run full 5-minute extraction at 4fps

Expected time: 40-60 minutes on T4 GPU.

Saves a raw CSV immediately after detection completes (recovery point if downstream steps fail). Then enriches with UTC timestamps and saves the final CSV.

In [ ]:
import time
import cv2
from datetime import datetime, timezone, timedelta
from pathlib import Path

# Configuration matching local pipeline
TARGET_FPS = 4
OUTPUT_NAME = 'facial_au_5min_colab.csv'
OUTPUT_PATH = Path(SESSION_DIR) / OUTPUT_NAME
RAW_PATH = OUTPUT_PATH.with_suffix('.raw.csv')

# Probe video
cap = cv2.VideoCapture(VIDEO_PATH)
native_fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
native_duration = total_frames / native_fps if native_fps > 0 else 0
cap.release()

skip_frames = max(1, int(round(native_fps / TARGET_FPS)))
effective_fps = native_fps / skip_frames
n_frames_to_process = total_frames // skip_frames

print(f'Video:                  {VIDEO_PATH}')
print(f'Native FPS:             {native_fps:.1f}')
print(f'Total frames:           {total_frames}')
print(f'Native duration:        {native_duration:.1f}s ({native_duration/60:.1f}m)')
print(f'Target FPS:             {TARGET_FPS}')
print(f'Skip frames:            {skip_frames} (effective {effective_fps:.2f} fps)')
print(f'Frames to process:      ~{n_frames_to_process}')
print(f'Output:                 {OUTPUT_PATH}')
print()

print('Running detection...')
start_time = time.monotonic()

fex = detector.detect_video(
    video_path=VIDEO_PATH,
    skip_frames=skip_frames,
    batch_size=1,
)

elapsed = time.monotonic() - start_time
print(f'\nDetection complete in {elapsed:.0f}s ({elapsed/60:.1f}m)')
print(f'Frames analyzed: {len(fex)}')

# Save raw FIRST (recovery point)
print(f'\nSaving raw output: {RAW_PATH}')
fex.to_csv(RAW_PATH, index=False)
print(f'  Raw size: {RAW_PATH.stat().st_size / (1024*1024):.1f} MB')

# Enrich with UTC timestamps
# Recording start parsed from session directory name (YYYYMMDDTHHMMSSZ)
session_name = Path(SESSION_DIR).name
ts_part = session_name.rsplit('_', 1)[-1]
recording_start = datetime.strptime(ts_part, '%Y%m%dT%H%M%SZ').replace(tzinfo=timezone.utc)
print(f'\nRecording start (UTC): {recording_start.isoformat()}')

clip_offset = 0  # No --start used for full extraction

def _to_utc(t):
    try:
        secs = float(t)
    except (ValueError, TypeError):
        parts = str(t).split(':')
        if len(parts) == 2:
            secs = int(parts[0]) * 60 + float(parts[1])
        elif len(parts) == 3:
            secs = int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
        else:
            return None
    return (recording_start + timedelta(seconds=secs + clip_offset)).isoformat()

if 'approx_time' in fex.columns:
    fex = fex.copy()
    fex['timestamp_utc'] = fex['approx_time'].apply(_to_utc)

print(f'\nWriting final CSV: {OUTPUT_PATH}')
fex.to_csv(OUTPUT_PATH, index=False)
size_mb = OUTPUT_PATH.stat().st_size / (1024*1024)
print(f'  Final size: {size_mb:.1f} MB')
print(f'  Rows: {len(fex)}')
print(f'  Columns: {len(fex.columns)}')
print(f'\nExtraction complete. Performance: {elapsed/len(fex):.2f}s per frame')

## Cell 6: Quick sanity check on output

Compare to expected: head pose, AUs, emotion probabilities.

In [ ]:
import pandas as pd

df = pd.read_csv(OUTPUT_PATH)
print(f'Shape: {df.shape}')
print(f'Columns: {len(df.columns)} total')
print(f'\nFace detection success rate: {df["FaceScore"].notna().sum()}/{len(df)} ({df["FaceScore"].notna().mean()*100:.1f}%)')
print(f'Mean FaceScore: {df["FaceScore"].mean():.3f}')

print('\n--- Head pose summary ---')
for axis in ['Pitch', 'Roll', 'Yaw']:
    print(f'  {axis:<6} {df[axis].mean():+.1f}° ± {df[axis].std():.1f}°')

print('\n--- AU mean intensities ---')
au_cols = [c for c in df.columns if c.startswith('AU') and not c.startswith('AU_')]
for au in au_cols:
    print(f'  {au}: {df[au].mean():.2f} ± {df[au].std():.2f}')

print('\n--- Emotion summary ---')
for emo in ['anger', 'disgust', 'fear', 'happiness', 'sadness', 'surprise', 'neutral']:
    if emo in df.columns:
        print(f'  {emo:<12} {df[emo].mean():.3f} ± {df[emo].std():.3f}')

print('\n--- Time range ---')
if 'timestamp_utc' in df.columns:
    print(f'  First: {df["timestamp_utc"].iloc[0]}')
    print(f'  Last:  {df["timestamp_utc"].iloc[-1]}')

## Cell 7: Optional — download the CSV locally

The file is already in Drive, but you can also download it directly through the browser.

In [ ]:
from google.colab import files
files.download(str(OUTPUT_PATH))